# Hinglish ASR: Whisper LoRA fine-tuning on MUCS 2021 (Colab or Kaggle, T4)
**Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. Keep the tab open. Checkpoints and results go to Google Drive
(`MyDrive/hinglish-cs-asr/`), so if the session disconnects just **Run all** again: training resumes from the last checkpoint.

**Kaggle:** Accelerator **GPU T4 x1**, Internet **On**, **Save Version → Save & Run All (Commit)**; download `hinglish_<RUN>.zip` from the Output tab.

Set the run in the next cell. Every step checks for errors and stops with a message if something fails.

In [ ]:
# ---- configuration ----
RUN   = 'turbo_r32'                           # name of this experiment
MODEL = 'openai/whisper-large-v3-turbo'
TRAIN_ARGS = '--rank 32 --targets both --epochs 2 --bs 16 --lr 5e-4'
SAVE_STEPS = 250  # checkpoint often: a Colab disconnect loses at most this many steps
SMOKE = False   # True: 30 training steps on 200 utterances + eval on 100 utterances (~15 min) to test the pipeline


In [ ]:
import os, subprocess, sys
def sh(cmd):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-3000:]); print(r.stderr[-3000:], file=sys.stderr)
    if r.returncode != 0: raise RuntimeError(f'FAILED ({r.returncode}): {cmd}')
def stream(cmd):  # live output for long jobs
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout: print(line, end='', flush=True)
    if p.wait() != 0: raise RuntimeError(f'FAILED ({p.returncode}): {cmd}')
COLAB = 'google.colab' in sys.modules
WORK = '/content' if COLAB else '/kaggle/working'  # Colab images can also contain a /kaggle folder
if COLAB:  # persist checkpoints + results on Drive so a disconnect does not lose the run
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST = '/content/drive/MyDrive/hinglish-cs-asr'
else:
    PERSIST = WORK
os.makedirs(PERSIST, exist_ok=True)
os.chdir(WORK)
if subprocess.run('nvidia-smi', shell=True, capture_output=True).returncode != 0:
    raise RuntimeError('NO GPU on this runtime. Colab: Runtime -> Change runtime type -> T4 GPU, then Run all. '
                       'Kaggle: Settings -> Accelerator -> GPU T4 x1.')
sh(f'nvidia-smi --query-gpu=name,memory.total --format=csv; df -h /tmp {WORK} | cat')
if not os.path.exists('hinglish-cs-asr'):
    sh('git clone -q https://github.com/hamzak-27/hinglish-cs-asr.git')
os.chdir('hinglish-cs-asr'); sh('git pull -q; git log --oneline | head -3')
sh('pip -q install "transformers>=4.45,<5" peft accelerate faster-whisper ctranslate2 jiwer soundfile nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"')


In [ ]:
# Make the pip-installed CUDA libs visible to CTranslate2, then a GPU smoke test before spending time on data
import glob, site
libdirs = [os.path.dirname(p) for sp in site.getsitepackages() for p in glob.glob(sp + '/nvidia/*/lib/*.so*')]
os.environ['LD_LIBRARY_PATH'] = ':'.join(sorted(set(libdirs))) + ':' + os.environ.get('LD_LIBRARY_PATH', '')
open('/tmp/gpu_smoke.py', 'w').write('''
import numpy as np, torch
from faster_whisper import WhisperModel
print("torch", torch.__version__, "| built for CUDA", torch.version.cuda, "| cuda available:", torch.cuda.is_available())
m = WhisperModel("tiny", device="cuda", compute_type="float16")
print(list(m.transcribe(np.zeros(16000 * 3, dtype=np.float32))[0]))
print("GPU OK")
''')
sh('python /tmp/gpu_smoke.py')


In [ ]:
# MUCS 2021 Hindi-English (OpenSLR 104, CC BY-SA 4.0). ~10 GB of audio -> /tmp (Kaggle's working dir is capped at 20 GB)
MIRROR = 'https://openslr.elda.org/resources/104'
sh('mkdir -p /tmp/mucs data')
for part in ['test', 'train']:
    if not os.path.exists(f'/tmp/mucs/{part}/transcripts/text'):
        stream(f'curl -sSL --retry 5 {MIRROR}/Hindi-English_{part}.tar.gz | tar -xz -C /tmp/mucs')
# blind set: audio + references + the official scorer (inference/compute_wer.py, hin2engwords.txt)
if not os.path.exists('/tmp/mucs/subtask2_blindtest_wReadme/hindi/files/text'):
    stream(f'curl -sSL --retry 5 {MIRROR}/subtask2_blindtest_wReadme.tar.gz | tar -xz -C /tmp/mucs --exclude="*/bengali/*"')
sh('ln -sfn /tmp/mucs data/raw && ls data/raw && wc -l data/raw/*/transcripts/text data/raw/subtask2_blindtest_wReadme/hindi/files/text')


In [ ]:
# Train (resumes from the last checkpoint if re-run)
OUT = f'{PERSIST}/out/{RUN}' if COLAB else f'/tmp/out/{RUN}'
if SMOKE: OUT += '_smoke'
extra = '--max_steps 30 --limit 200' if SMOKE else ''
extra += f' --save_steps {SAVE_STEPS}'
stream(f'cd src && python train_lora.py --model {MODEL} --train_dir ../data/raw/train --out {OUT} {TRAIN_ARGS} {extra}')


In [ ]:
# Export to CTranslate2 (float16; faster-whisper can also run it as int8 on CPU), then evaluate.
# blind = the MUCS 2021 leaderboard set (official scorer); test = official test set.
CT2 = f'{PERSIST}/{RUN}{"_smoke" if SMOKE else ""}_ct2'
if not os.path.exists(f'{CT2}/model.bin'):
    sh(f'ct2-transformers-converter --model {OUT}/merged --output_dir {CT2} --quantization float16 '
       f'--copy_files tokenizer.json preprocessor_config.json --force')
if os.path.exists(f'{PERSIST}/results') and PERSIST != WORK:  # restore finished utterances after a disconnect
    sh(f'mkdir -p results && cp -rn {PERSIST}/results/. results/')
BASE = MODEL.split('/')[-1].replace('whisper-', '')
runs = [(CT2, RUN, 'hi', s, '--subset 100' if SMOKE else '') for s in (['test'] if SMOKE else ['blind', 'test'])]
if not SMOKE:  # zero-shot references for the same base model and decoding
    runs += [(BASE, '', 'hi', s, '') for s in ['blind', 'test']]
    runs += [(BASE, '', l, 'test', '--subset 500') for l in ['en', 'auto']]
for model, tag, lang, split, sub in runs:
    stream(f'cd src && python run_asr.py --model {model} {"--tag " + tag if tag else ""} --split {split} --lang {lang} '
           f'--device cuda --compute float16 {sub}')
    sh(f'mkdir -p {PERSIST}/results && cp -r results/{split} {PERSIST}/results/')  # results survive a disconnect
for split in ['blind', 'test']:
    if os.path.exists(f'results/{split}'): sh(f'cd src && python evaluate.py {split}')


In [ ]:
# Package: results + adapter + training log + CTranslate2 model
sh(f'cp -r {OUT}/adapter {OUT}/train_args.json {WORK}/ 2>/dev/null; ls {OUT}')
sh(f'cp $(ls -d {OUT}/checkpoint-* | tail -1)/trainer_state.json {WORK}/{RUN}_trainer_state.json')
sh(f'cp -r {CT2} {WORK}/{RUN}_ct2 2>/dev/null || true')
sh(f'cd {WORK} && zip -qr hinglish_{RUN}.zip hinglish-cs-asr/results adapter train_args.json {RUN}_trainer_state.json {RUN}_ct2')
if COLAB: sh(f'cp {WORK}/hinglish_{RUN}.zip {PERSIST}/')
print('Download:', f'{PERSIST}/hinglish_{RUN}.zip' if COLAB else f'{WORK}/hinglish_{RUN}.zip')
